## Limpieza de los datos filtrados del SENA

Este notebook limpia y alista los datos descargados desde Athena para que puedan
ser utilizados posteriormente en los análisis de las tres preguntas de negocio.

### Librerías

In [37]:
import numpy as np
import pandas as pd

### Carga de datos

In [38]:
ARCHIVO_CSV = "datos_filtrados_raw.csv"
df = pd.read_csv(ARCHIVO_CSV)

### Vista inicial

In [39]:
df.head()

,meta_id,meta_version,meta_created_at,meta_updated_at,nombre_entidad,nit_entidad,departamento,ciudad,localizaci_n,orden,...,n_mero_de_documento_ordenador_del_gasto,nombre_supervisor,tipo_de_documento_supervisor,n_mero_de_documento_supervisor,nombre_ordenador_de_pago,tipo_de_documento_ordenador_de_pago,n_mero_de_documento_ordenador_de_pago,documentos_tipo,descripcion_documentos_tipo,direcci_n_de_ejecuci_n_del_contrato
0,row-mrjp.c93w~a6c6,rv-mcdr-jfsq.d9mx,2026-08-23T08:01:59.248Z,2026-08-23T08:01:59.248Z,SENA REGIONAL VALLE Grupo de Apoyo Administrat...,899999034,Valle del Cauca,Cali,"Colombia, Valle del Cauca, Cali",Nacional,...,31982003,Nelson Ortega Valdes,Cédula de Ciudadanía,6247216,No definido,No definido,No definido,No,No definido,CALLE 52 # 2 BIS - 15\nCali\nValle del Cauca\n...
1,row-9piy_2569-3rzc,rv-9i3g~d2vv~hrmf,2026-08-23T08:01:59.248Z,2026-08-23T08:01:59.248Z,SENA REGIONAL VALLE Grupo de Apoyo Administrat...,899999034,Valle del Cauca,Cali,"Colombia, Valle del Cauca, Cali",Nacional,...,94386599,Jorge Humberto Peña,Cédula de Ciudadanía,16658614,No definido,No definido,No definido,No,No definido,CALLE 52 # 2 BIS - 15\nCali\nValle del Cauca\n...
2,row-5edk-jwx5_givt,rv-xzps~6uev-29w5,2026-08-23T08:01:59.248Z,2026-08-23T08:01:59.248Z,SENA REGIONAL ANTIOQUIA Grupo Administrativo CIAA,899999034,Antioquia,Rionegro,"Colombia, Antioquia , Rionegro",Nacional,...,71608941,LUIS UBEIRMAR VALENCIA AGUDELO,Cédula de Ciudadanía,70907307,No definido,No definido,No definido,No,No definido,DE CONFORMIDAD CON LAS ESPECIFICACIONES TECNIC...
3,row-eatr.7e9n.7rvx,rv-f3f2_x36v_dkgm,2026-08-23T08:01:59.248Z,2026-08-23T08:01:59.248Z,SENA Regional Huila Grupo de Apoyo Administrat...,899999034,Huila,Neiva,"Colombia, Huila, Neiva",Nacional,...,83115849,CESAR AUGUSTO PERDOMO FUENTES,Cédula de Ciudadanía,1075209543,Fermín Beltrán Barragán,Cédula de Ciudadanía,83115849,No,No definido,Carrera 5 No. 16-16\nNeiva\nHuila\nCOLOMBIA
4,row-irah.a3wm~kviu,rv-6423~zue5~hjhc,2026-08-23T08:01:59.248Z,2026-08-23T08:01:59.248Z,SENA REGIONAL TOLIMA 1,899999034,Tolima,Ibagué,"Colombia, Tolima , Ibagué",Nacional,...,1053783162,Alvaro Eliu Melo Hernandez,Cédula de Ciudadanía,93409341,No definido,No definido,No definido,No,No definido,CARRERA 45 SUR No. 141-05 SECTOR PICALEÑA \nIb...


### Dimensiones iniciales

In [40]:
df.shape

(328057, 89)

### Selección de columnas

Se conservan únicamente las variables necesarias para responder las preguntas de negocio
y para caracterizar correctamente los contratos.

In [41]:
columnas_utiles = [
    # Entidad y ubicación
    "nombre_entidad",
    "nit_entidad",
    "codigo_entidad",
    "departamento",
    "ciudad",

    # Identificación del contrato
    "proceso_de_compra",
    "id_contrato",
    "estado_contrato",

    # Tipo y categoría del contrato
    "codigo_de_categoria_principal",
    "descripcion_del_proceso",
    "tipo_de_contrato",
    "modalidad_de_contratacion",
    "objeto_del_contrato",

    # Fechas
    "fecha_de_firma",
    "fecha_de_inicio_del_contrato",
    "fecha_de_fin_del_contrato",

    # Proveedor
    "tipodocproveedor",
    "documento_proveedor",
    "proveedor_adjudicado",
    "es_grupo",

    # Valores monetarios
    "valor_del_contrato",
    "valor_pagado",

    # Duración y prórrogas
    "dias_adicionados",
    "duraci_n_del_contrato",
    "el_contrato_puede_ser_prorrogado",
    "fecha_de_notificaci_n_de_prorrogaci_n"
]

df = df[columnas_utiles].copy()

### Vista de las columnas seleccionadas

In [42]:
df.head()

,nombre_entidad,nit_entidad,codigo_entidad,departamento,ciudad,proceso_de_compra,id_contrato,estado_contrato,codigo_de_categoria_principal,descripcion_del_proceso,...,tipodocproveedor,documento_proveedor,proveedor_adjudicado,es_grupo,valor_del_contrato,valor_pagado,dias_adicionados,duraci_n_del_contrato,el_contrato_puede_ser_prorrogado,fecha_de_notificaci_n_de_prorrogaci_n
0,SENA REGIONAL VALLE Grupo de Apoyo Administrat...,899999034,700763170,Valle del Cauca,Cali,CO1.BDOS.1674748,CO1.PCCNTR.2146739,terminado,V1.80111600,Prestar los servicios como Profesional Regiona...,...,Cédula de Ciudadanía,94362177,Raul Orlando Tamayo Apolindar,No,42558004.0,42558004,0,340 Dia(s),No,NaN
1,SENA REGIONAL VALLE Grupo de Apoyo Administrat...,899999034,700763170,Valle del Cauca,Cali,CO1.BDOS.2546395,CO1.PCCNTR.3280659,Cerrado,V1.80111600,Prestar los servicios profesionales para apoya...,...,Cédula de Ciudadanía,1144034093,GRISALES CASTAÑO LAURA VICTORIA,No,39790000.0,39790000,0,345 Dia(s),No,NaN
2,SENA REGIONAL ANTIOQUIA Grupo Administrativo CIAA,899999034,704160423,Antioquia,Rionegro,CO1.BDOS.1741951,CO1.PCCNTR.2229934,Cerrado,V1.80111600,5_9503_184 Prestar servicios personales de car...,...,Cédula de Ciudadanía,43904988,MARGARITA MARIA MESA AGUDELO,No,40170000.0,40170000,0,309 Dia(s),Si,2021-11-08T00:00:00.000
3,SENA Regional Huila Grupo de Apoyo Administrat...,899999034,702689464,Huila,Neiva,CO1.BDOS.9130377,CO1.PCCNTR.8650729,En ejecución,V1.90111503,41_9527_587 Contratar el servicio de operador ...,...,Cédula de Ciudadanía,1075249833,CARLOS MAURICIO HOYOS PUCCINY,No,16800000.0,0,0,15 Dia(s),Si,2025-12-08T00:00:00.000
4,SENA REGIONAL TOLIMA 1,899999034,702561986,Tolima,Ibagué,CO1.BDOS.1729110,CO1.PCCNTR.2228398,Modificado,V1.80111600,Prestación de servicios de apoyo a la gestión ...,...,Cédula de Ciudadanía,5821261,jhon jairo riveros lugo,No,31750500.0,31750500,5,309 Dia(s),Si,NaN


### Dimensiones después de seleccionar columnas

In [43]:
df.shape

(328057, 26)

### Datos faltantes

In [44]:
df.isna().sum()

nombre_entidad                                0
nit_entidad                                   0
codigo_entidad                                0
departamento                                  0
ciudad                                        0
proceso_de_compra                             0
id_contrato                                   0
estado_contrato                               0
codigo_de_categoria_principal                 0
descripcion_del_proceso                       0
tipo_de_contrato                              0
modalidad_de_contratacion                     0
objeto_del_contrato                           0
fecha_de_firma                            16624
fecha_de_inicio_del_contrato              16589
fecha_de_fin_del_contrato                  1729
tipodocproveedor                              0
documento_proveedor                           0
proveedor_adjudicado                          1
es_grupo                                      0
valor_del_contrato                      

### Porcentaje de datos faltantes

In [45]:
df.isna().sum() / df.shape[0] * 100

nombre_entidad                            0.000000
nit_entidad                               0.000000
codigo_entidad                            0.000000
departamento                              0.000000
ciudad                                    0.000000
proceso_de_compra                         0.000000
id_contrato                               0.000000
estado_contrato                           0.000000
codigo_de_categoria_principal             0.000000
descripcion_del_proceso                   0.000000
tipo_de_contrato                          0.000000
modalidad_de_contratacion                 0.000000
objeto_del_contrato                       0.000000
fecha_de_firma                            5.067412
fecha_de_inicio_del_contrato              5.056743
fecha_de_fin_del_contrato                 0.527043
tipodocproveedor                          0.000000
documento_proveedor                       0.000000
proveedor_adjudicado                      0.000305
es_grupo                       

Los faltantes se concentran principalmente en variables de fecha. No se eliminan ni
se imputan estos valores, porque reemplazarlos implicaría introducir información que
no está presente de base. Los análisis posteriores utilizarán los registros
disponibles según las variables que requieran.

### Tipos de datos

In [46]:
df.dtypes

nombre_entidad                               str
nit_entidad                                int64
codigo_entidad                             int64
departamento                                 str
ciudad                                       str
proceso_de_compra                            str
id_contrato                                  str
estado_contrato                              str
codigo_de_categoria_principal                str
descripcion_del_proceso                      str
tipo_de_contrato                             str
modalidad_de_contratacion                    str
objeto_del_contrato                          str
fecha_de_firma                               str
fecha_de_inicio_del_contrato                 str
fecha_de_fin_del_contrato                    str
tipodocproveedor                             str
documento_proveedor                          str
proveedor_adjudicado                         str
es_grupo                                     str
valor_del_contrato  

In [47]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 328057 entries, 0 to 328056
Data columns (total 26 columns):
 #   Column                                 Non-Null Count   Dtype  
---  ------                                 --------------   -----  
 0   nombre_entidad                         328057 non-null  str    
 1   nit_entidad                            328057 non-null  int64  
 2   codigo_entidad                         328057 non-null  int64  
 3   departamento                           328057 non-null  str    
 4   ciudad                                 328057 non-null  str    
 5   proceso_de_compra                      328057 non-null  str    
 6   id_contrato                            328057 non-null  str    
 7   estado_contrato                        328057 non-null  str    
 8   codigo_de_categoria_principal          328057 non-null  str    
 9   descripcion_del_proceso                328057 non-null  str    
 10  tipo_de_contrato                       328057 non-null  str    
 11

### Estadísticas descriptivas

In [48]:
df.describe()

,nit_entidad,codigo_entidad,valor_del_contrato,valor_pagado,dias_adicionados
count,3.280570e+05,3.280570e+05,3.280570e+05,3.280570e+05,328057.000000
mean,1.325224e+09,7.040945e+08,6.238513e+11,3.861985e+07,3.077712
std,1.806520e+09,2.896096e+06,5.540689e+13,2.016532e+09,18.765240
min,8.999990e+08,7.000980e+08,0.000000e+00,0.000000e+00,0.000000
25%,8.999990e+08,7.029884e+08,1.998607e+07,8.843683e+06,0.000000
50%,8.999990e+08,7.041560e+08,3.550762e+07,2.605200e+07,0.000000
75%,8.999990e+08,7.041614e+08,4.538184e+07,3.997500e+07,0.000000
max,8.999990e+09,7.248610e+08,2.025988e+16,1.088472e+12,1833.000000


### Valores monetarios extremos

In [49]:
df.nlargest(
    10,
    "valor_del_contrato"
)[
    [
        "nombre_entidad",
        "id_contrato",
        "tipo_de_contrato",
        "valor_del_contrato",
        "valor_pagado"
    ]
]

,nombre_entidad,id_contrato,tipo_de_contrato,valor_del_contrato,valor_pagado
172862,SENA SECRETARIA GENERAL,CO1.PCCNTR.8826739,Prestación de servicios,2.025988e+16,0
128742,SENA REGIONAL ATLANTICO,CO1.PCCNTR.7261206,Prestación de servicios,1.182376e+16,0
58282,SENA REGIONAL VICHADA,CO1.PCCNTR.6372301,Prestación de servicios,7.152534e+15,0
237540,SENA REGIONAL VICHADA,CO1.PCCNTR.6372303,Prestación de servicios,6.826050e+15,0
237596,SENA REGIONAL VICHADA,CO1.PCCNTR.6372305,Prestación de servicios,6.684101e+15,0
325730,SENA REGIONAL VICHADA,CO1.PCCNTR.6372304,Prestación de servicios,5.619049e+15,0
43578,SENA REGIONAL VICHADA,CO1.PCCNTR.6372302,Prestación de servicios,5.419030e+15,0
246168,SENA REGIONAL SANTANDER Grupo Administrativo CA,CO1.PCCNTR.8825896,Prestación de servicios,5.358547e+15,0
68374,SENA REGIONAL TOLIMA 1,CO1.PCCNTR.5787958,Prestación de servicios,4.147360e+15,0
168198,SENA SECRETARIA GENERAL,CO1.PCCNTR.751286,Prestación de servicios,3.928545e+15,0


Se identifican valores extremos en valor_del_contrato. No se eliminan automáticamente,
ya que antes sería necesario validar cada caso con el contexto de la base de datos original.

### Duplicados

In [50]:
df.duplicated().sum()

np.int64(0)

In [51]:
df.drop_duplicates().shape

(328057, 26)

No se encontraron duplicados exactos en las variables seleccionadas, por lo que no es
necesario eliminar registros por este criterio.

### Conversión de fechas

In [52]:
columnas_fechas = [
    "fecha_de_firma",
    "fecha_de_inicio_del_contrato",
    "fecha_de_fin_del_contrato",
    "fecha_de_notificaci_n_de_prorrogaci_n"
]

for columna in columnas_fechas:
    df[columna] = pd.to_datetime(df[columna], errors="coerce")

In [53]:
df[columnas_fechas].dtypes

fecha_de_firma                           datetime64[us]
fecha_de_inicio_del_contrato             datetime64[us]
fecha_de_fin_del_contrato                datetime64[us]
fecha_de_notificaci_n_de_prorrogaci_n    datetime64[us]
dtype: object

### Identificadores como texto

In [54]:
df["nit_entidad"] = df["nit_entidad"].astype(str)
df["codigo_entidad"] = df["codigo_entidad"].astype(str)
df["documento_proveedor"] = df["documento_proveedor"].astype(str)

Los identificadores se almacenan como texto porque representan códigos y documentos,
no cantidades sobre las que tenga sentido realizar operaciones matemáticas.

### Año y mes de firma

In [55]:
df["anio_firma"] = df["fecha_de_firma"].dt.year.astype("Int64")
df["mes_firma"] = df["fecha_de_firma"].dt.month.astype("Int64")

### Duración del contrato

In [56]:
df["duracion_dias"] = (
    df["fecha_de_fin_del_contrato"]
    - df["fecha_de_inicio_del_contrato"]
).dt.days

df["duracion_dias"].describe()

count    311464.000000
mean        249.294512
std         145.015509
min        -348.000000
25%         165.000000
50%         302.000000
75%         319.000000
max        4000.000000
Name: duracion_dias, dtype: float64

### Duraciones inconsistentes

In [57]:
(df["duracion_dias"] < 0).sum()

np.int64(9)

In [58]:
df.loc[
    df["duracion_dias"] < 0,
    [
        "id_contrato",
        "fecha_de_inicio_del_contrato",
        "fecha_de_fin_del_contrato",
        "duracion_dias"
    ]
]

,id_contrato,fecha_de_inicio_del_contrato,fecha_de_fin_del_contrato,duracion_dias
43768,CO1.PCCNTR.8708943,2025-12-23,2025-12-22,-1.0
69581,CO1.PCCNTR.8581220,2025-12-05,2025-11-30,-5.0
86786,CO1.PCCNTR.8917121,2026-01-17,2025-12-31,-17.0
90963,CO1.PCCNTR.8584964,2025-11-25,2024-12-12,-348.0
138171,CO1.PCCNTR.1237319,2019-12-25,2019-11-25,-30.0
175815,CO1.PCCNTR.8409201,2025-11-14,2025-10-26,-19.0
221504,CO1.PCCNTR.1350659,2020-02-15,2020-02-06,-9.0
259928,CO1.PCCNTR.8601738,2025-11-27,2024-12-16,-346.0
275616,CO1.PCCNTR.8915244,2026-01-16,2026-01-15,-1.0


Las duraciones negativas corresponden a registros cuya fecha de fin es anterior a la
fecha de inicio. Se conservan las fechas originales para trazabilidad, pero la duración
calculada se marca como faltante para evitar utilizar un valor imposible en análisis
posteriores.

In [59]:
df["duracion_inconsistente"] = df["duracion_dias"] < 0
df.loc[df["duracion_inconsistente"], "duracion_dias"] = np.nan

### Código UNSPSC

In [60]:
df["unspsc"] = (
    df["codigo_de_categoria_principal"]
    .str.extract(r"(\d{8})", expand=False)
)

df[["codigo_de_categoria_principal", "unspsc"]].head()

,codigo_de_categoria_principal,unspsc
0,V1.80111600,80111600
1,V1.80111600,80111600
2,V1.80111600,80111600
3,V1.90111503,90111503
4,V1.80111600,80111600


### Regional

In [61]:
df["regional"] = df["departamento"]

nivel_central = (
    df["nombre_entidad"]
    .str.contains(
        "SECRETARIA GENERAL|DIRECCION GENERAL",
        case=False,
        na=False
    )
)

df.loc[nivel_central, "regional"] = "Nivel Central"

In [62]:
df["regional"].value_counts()

regional
Distrito Capital de Bogotá                  48002
Antioquia                                   36113
Valle del Cauca                             23212
Atlántico                                   19554
Santander                                   19172
Cundinamarca                                13177
Nivel Central                               11799
Huila                                       10890
Cauca                                       10591
Tolima                                      10507
Bolívar                                     10427
Boyacá                                       9807
Norte de Santander                           9401
Cesar                                        8650
Córdoba                                      8104
Caldas                                       8035
Nariño                                       7702
Risaralda                                    7581
Quindío                                      6776
Magdalena                                

### Prórrogas

In [63]:
df["el_contrato_puede_ser_prorrogado"].value_counts(dropna=False)

el_contrato_puede_ser_prorrogado
No    257052
Si     71005
Name: count, dtype: int64

In [64]:
prorroga = (
    df["el_contrato_puede_ser_prorrogado"]
    .str.strip()
    .str.lower()
)

df["es_prorrogable"] = prorroga.map({
    "si": True,
    "sí": True,
    "no": False
}).astype("boolean")

df["tiene_notificacion_prorroga"] = (
    df["fecha_de_notificaci_n_de_prorrogaci_n"].notna()
)

### Categorías para identificar contratación individual

In [65]:
df["tipo_de_contrato"].value_counts()

tipo_de_contrato
Prestación de servicios           300960
Compraventa                        14437
Suministros                         6602
Otro                                2921
Obra                                1377
Arrendamiento de inmuebles          1112
Consultoría                          257
Interventoría                        135
Decreto 092 de 2017                  133
Arrendamiento de muebles              35
Seguros                               34
Asociación Público Privada            21
Comodato                              10
Servicios financieros                 10
Comisión                               4
No Especificado                        3
Negocio fiduciario                     2
Venta muebles                          1
Operaciones de Crédito Público         1
Venta inmuebles                        1
Concesión                              1
Name: count, dtype: int64

In [66]:
df["tipodocproveedor"].value_counts()

tipodocproveedor
Cédula de Ciudadanía               286405
NIT                                 33260
No Definido                          7416
Otro                                  609
Cédula de Extranjería                 319
Permiso por Protección Temporal        23
Tarjeta de Identidad                   14
Registro Civil                          7
Pasaporte                               4
Name: count, dtype: int64

In [67]:
df["es_grupo"].value_counts()

es_grupo
No    327257
Si       800
Name: count, dtype: int64

Para dejar una definición común para los tres análisis, se considera contratación
individual de personal a los contratos de prestación de servicios adjudicados a una
persona con un documento de identificación personal y que no estén marcados como grupo.

### Contratación individual de personal

In [68]:
documentos_persona = [
    "Cédula de Ciudadanía",
    "Cédula de Extranjería",
    "Permiso por Protección Temporal",
    "Tarjeta de Identidad",
    "Registro Civil",
    "Pasaporte"
]

df["es_contratacion_individual"] = (
    (df["tipo_de_contrato"] == "Prestación de servicios")
    & (df["tipodocproveedor"].isin(documentos_persona))
    & (df["es_grupo"] == "No")
)

df["es_contratacion_individual"].value_counts()

es_contratacion_individual
True     281635
False     46422
Name: count, dtype: int64

### Validación final

In [69]:
df.shape

(328057, 35)

In [70]:
df.isna().sum()

nombre_entidad                                0
nit_entidad                                   0
codigo_entidad                                0
departamento                                  0
ciudad                                        0
proceso_de_compra                             0
id_contrato                                   0
estado_contrato                               0
codigo_de_categoria_principal                 0
descripcion_del_proceso                       0
tipo_de_contrato                              0
modalidad_de_contratacion                     0
objeto_del_contrato                           0
fecha_de_firma                            16624
fecha_de_inicio_del_contrato              16589
fecha_de_fin_del_contrato                  1729
tipodocproveedor                              0
documento_proveedor                           0
proveedor_adjudicado                          1
es_grupo                                      0
valor_del_contrato                      

In [71]:
df.dtypes

nombre_entidad                                      str
nit_entidad                                         str
codigo_entidad                                      str
departamento                                        str
ciudad                                              str
proceso_de_compra                                   str
id_contrato                                         str
estado_contrato                                     str
codigo_de_categoria_principal                       str
descripcion_del_proceso                             str
tipo_de_contrato                                    str
modalidad_de_contratacion                           str
objeto_del_contrato                                 str
fecha_de_firma                           datetime64[us]
fecha_de_inicio_del_contrato             datetime64[us]
fecha_de_fin_del_contrato                datetime64[us]
tipodocproveedor                                    str
documento_proveedor                             

### Exportar datos limpios

Se exporta un archivo con todos los contratos SENA limpiados y otro con únicamente
la contratación individual de personal, que será la base principal para las tres
preguntas de negocio.

In [72]:
df.to_csv(
    "datos_limpios.csv",
    index=False,
    encoding="utf-8-sig"
)